In [14]:
# requirements:
# %pip install requests beautifulsoup4 lxml tqdm By
%pip install rembg pillow numpy tqdm

  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl (38.9 MB)
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/15.6 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 4.2/15.6 MB 11.4 MB/s eta 0:00:01
   -------------- ------------------------- 5.8/15.6 MB 9.8 MB/s eta 0:00:02
   ------------------ --------------------- 7.1/15.6 MB 9.1 MB/s eta 0:00:01
   ---------------------- ----------------- 8.7/15.6 MB 8.7 MB/s eta 0:00:01
   -------------------------- ------------- 10.2/15.6 MB 8.5 MB/s eta 0:00:01
   ------------------------------ --------- 11.8/15.6 MB 8.3 MB/s eta 0:00:01
   ---------------------------------- ----- 13.4/15.6 MB 8.1 MB/s eta 0:00:01
   -------------------------------------- - 14.9/15.6 MB 8.0 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.


In [ ]:
# requirements:
#   pip install selenium webdriver-manager requests tqdm

import os, re, time, random
from pathlib import Path
from urllib.parse import urljoin, urlparse
import requests
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# from webdriver_manager.chrome import ChromeDriverManager  # (Selenium Manager 사용 시 불필요)

BASE_URL  = "https://www.hyundai.com"    # 최종 이미지에 붙일 도메인
START_URL = "https://www.hyundai.com/kr/ko/e/menu-list/"   # #model 이 있는 실제 페이지 URL
OUT_DIR   = Path("insight&trends_images")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": BASE_URL
}

# ---- Rate limit-friendly delays ----
REQ_DELAY_RANGE = (3.5, 7.0)   # 페이지 이동 사이 대기(랜덤)
IMG_DELAY_RANGE = (1.0, 2.2)   # 이미지 다운로드 사이 대기(랜덤)
BLOCK_PATTERNS = ["Error 1015", "You are being rate limited", "cf-error-code", "Ray ID"]

def polite_sleep(a, b):
    time.sleep(random.uniform(a, b))

def detect_block_html(html: str) -> bool:
    low = html.lower()
    return any(p.lower() in low for p in BLOCK_PATTERNS)

def polite_get(driver, url, wait, locator=None):
    polite_sleep(*REQ_DELAY_RANGE)
    driver.get(url)
    if locator:
        try:
            wait.until(EC.presence_of_element_located(locator))
        except:
            pass
    if detect_block_html(driver.page_source):
        raise RuntimeError(f"[BLOCK] Cloudflare rate limit 감지: {url}")

# -------- Utils --------
def extract_bg_url(style_text: str) -> str | None:
    """
    style="background-image: url('/contents/.../xxx.jpg');" 에서 /contents/... 추출
    url("..."), url('...'), url(...) 모두 대응
    """
    if not style_text:
        return None
    style_text = style_text.replace("&quot;", '"')
    m = re.search(r'background-image\s*:\s*url\((["\']?)([^)"\']+)\1\)', style_text, flags=re.I)
    if not m:
        m = re.search(r'url\((["\']?)([^)"\']+)\1\)', style_text, flags=re.I)
    return m.group(2) if m else None

def safe_filename(url: str) -> str:
    name = os.path.basename(urlparse(url).path) or "image"
    return re.sub(r'[^A-Za-z0-9._-]+', "_", name)

def download(url: str, referer: str | None = None):
    polite_sleep(*IMG_DELAY_RANGE)
    headers = dict(HEADERS)
    if referer:
        headers["Referer"] = referer
    with requests.get(url, headers=headers, stream=True, timeout=30) as r:
        r.raise_for_status()
        fname = safe_filename(url)
        dst = OUT_DIR / fname
        # 중복 파일명 방지
        i = 1
        while dst.exists():
            stem, ext = os.path.splitext(fname)
            dst = OUT_DIR / f"{stem}_{i}{ext}"
            i += 1
        total = int(r.headers.get("Content-Length", 0))
        with open(dst, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=fname, leave=False) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    return dst

# -------- Selenium setup --------
opts = Options()
opts.add_argument("--headless=new")      # 필요 시 주석 처리해서 브라우저 보이게
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1400,900")

driver = webdriver.Chrome(options=opts)  # Selenium 4.6+면 경로 자동관리
wait = WebDriverWait(driver, 15)

try:
    # ===== 1) #model 밑의 모든 li 수집 (nth-child 제거) =====
    polite_get(driver, START_URL, wait, (By.CSS_SELECTOR, "#model"))
    # 직계 li 전부 → 그 하위 li 전부
    items = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "#model > ul > li ul > li")
        )
    )
    if not items:
        raise RuntimeError("#model 하위에서 ul > li 항목을 찾지 못했습니다. 선택자/페이지를 확인하세요.")

    # ===== 2) 각 항목에서 a[href] 추출 (nth-child 사용 안 함) =====
    detail_links = []
    for li in items:
        try:
            a = li.find_element(By.CSS_SELECTOR, "a[href]")
            href = (a.get_attribute("href") or "").strip()
            if not href:
                continue
            abs_url = urljoin(BASE_URL, href)
            if abs_url not in detail_links:
                detail_links.append(abs_url)
        except:
            continue

    print(f"[INFO] 수집된 상세 링크: {len(detail_links)}개")

    # ===== 3) 상세 링크들에서 background-image 경로 추출 =====
    image_urls = []              # 절대 URL 리스트
    url_to_referer = {}          # 다운로드 시 Referer로 쓸 맵

    for link in tqdm(detail_links, desc="상세 페이지 파싱"):
        try:
            polite_get(driver, link, wait)  # 느슨히 로드 + 차단 감지
            # 기본 선택자
            divs = driver.find_elements(By.CSS_SELECTOR, "#carModelInfo > section.visual-wrap.design2023 > div")
            if not divs:
                # 폴백: 클래스 변형/구조 변경 대비
                divs = driver.find_elements(By.CSS_SELECTOR, "section.visual-wrap div[style*='background-image']")

            if not divs:
                print(f"[WARN] 배경이미지 div를 찾지 못함: {link}")
                continue

            found = False
            for div in divs:
                style = div.get_attribute("style") or ""
                rel = extract_bg_url(style)
                if not rel:
                    continue
                abs_img = urljoin(BASE_URL, rel)
                if abs_img not in url_to_referer:
                    image_urls.append(abs_img)
                    url_to_referer[abs_img] = link
                    found = True
            if not found:
                print(f"[WARN] style에서 url()을 추출하지 못함: {link}")

        except RuntimeError as e:
            # Cloudflare 1015 감지 시 종료
            print(str(e))
            break
        except Exception as e:
            print(f"[ERROR] {link}: {e}")

    image_urls = list(dict.fromkeys(image_urls))
    print(f"[INFO] 추출된 이미지 URL: {len(image_urls)}개")

    # ===== 4) 이미지 다운로드 (랜덤 지연 + Referer 포함) =====
    for img_url in tqdm(image_urls, desc="이미지 다운로드"):
        try:
            saved = download(img_url, referer=url_to_referer.get(img_url, BASE_URL))
            # print("saved:", saved)
        except Exception as e:
            print(f"[ERROR] download fail: {img_url} -> {e}")

    print(f"[DONE] 저장 폴더: {OUT_DIR.resolve()}")

finally:
    driver.quit()


[INFO] 수집된 상세 링크: 56개


상세 페이지 파싱:  29%|██▊       | 16/56 [02:39<07:13, 10.83s/it]

[WARN] 배경이미지 div를 찾지 못함: https://www.hyundai.com/kr/ko/e/vehicles/sonata-the-edge-hybrid/intro


상세 페이지 파싱:  54%|█████▎    | 30/56 [05:02<03:26,  7.96s/it]

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from rembg import remove
from tqdm import tqdm

SRC_DIR = Path("insight&trends_images")
OUT_DIR = Path("insight&trends_images_cropped")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PADDING_RATIO = 0.06  # 실루엣 바운딩박스 주변 여백

def crop_by_alpha(img_rgba: Image.Image, pad_ratio: float):
    """알파 채널을 기준으로 유효 영역 bbox 계산 후 패딩 포함 크롭"""
    if img_rgba.mode != "RGBA":
        img_rgba = img_rgba.convert("RGBA")
    arr = np.array(img_rgba)
    alpha = arr[:, :, 3]
    ys, xs = np.where(alpha > 0)  # 알파가 남은(배경 제거 안 된) 영역
    if len(xs) == 0 or len(ys) == 0:
        return img_rgba  # 전부 제거된 경우 원본 반환(혹은 skip)
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    h, w = alpha.shape
    bw, bh = (x2 - x1 + 1), (y2 - y1 + 1)
    px = int(bw * pad_ratio)
    py = int(bh * pad_ratio)

    nx1 = max(0, x1 - px)
    ny1 = max(0, y1 - py)
    nx2 = min(w, x2 + px + 1)
    ny2 = min(h, y2 + py + 1)

    return img_rgba.crop((nx1, ny1, nx2, ny2))

images = [p for p in SRC_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
print(f"[INFO] 대상 이미지: {len(images)}개")

for img_path in tqdm(images, desc="rembg crop"):
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGBA")
            # 배경 제거
            cut = remove(im)  # RGBA 반환(배경 투명)
            # 알파 유효 영역으로 크롭
            cropped = crop_by_alpha(cut, PADDING_RATIO)
            # 배경을 흰색으로 깔고 저장하고 싶다면 아래 3줄 사용
            # bg = Image.new("RGB", cropped.size, (255, 255, 255))
            # bg.paste(cropped, mask=cropped.split()[3])
            # bg.save(OUT_DIR / (img_path.stem + ".jpg"), quality=95)

            # 투명 유지하여 PNG로 저장
            out_ext = ".png" if cropped.mode == "RGBA" else img_path.suffix
            cropped.save(OUT_DIR / (img_path.stem + out_ext))
    except Exception as e:
        print(f"[ERROR] {img_path.name}: {e}")

print(f"[DONE] 저장 폴더: {OUT_DIR.resolve()}")

